# Oil Field Investment Optimization Using Predictive Modeling and Risk Simulation

Project Type: Machine Learning + Financial Risk Analysis
Tools: Python, Pandas, NumPy, Scikit-Learn, Bootstrapping Simulation
Model: Linear Regression
Business Focus: Capital Allocation Under Risk

## Data Preparation Phase

In [1]:
# To operate with DataFrames
import pandas as pd

# Importation of the NumPy library
import numpy as np

# For the generation of random values
from numpy.random import RandomState

# For categorical label encoding
from sklearn.preprocessing import OrdinalEncoder

# For dataset partitioning
from sklearn.model_selection import train_test_split

# For the implementation of linear regression models
from sklearn.linear_model import LinearRegression

# Mean Squared Error (MSE) evaluation metric
from sklearn.metrics import mean_squared_error

### Reading the DataFrames and Displaying Their Contents

In [2]:
df = []
for i in range(3):
    path = f"../data/geo_data_{i}.csv"
    df.append(pd.read_csv(path))

In [3]:
def printDfInfo():
    for i in range(len(df)):
        print("REGION{}".format(i))
        print(df[i].head())
        print("\ninfo")
        print(df[i].info())
        print("\n")

Our DataFrames do not contain missing values, and the data types appear to be appropriate. Prior to removing the ID column, I will verify whether duplicate rows are present. If no duplicate rows are identified, I will proceed with the elimination of the ID column.

In [4]:
for i in range(len(df)):
    print("FILAS DUPLICADAS EN REGION{}".format(i))
    print(df[i].duplicated().sum())
    print("\n")
    

FILAS DUPLICADAS EN REGION0
0


FILAS DUPLICADAS EN REGION1
0


FILAS DUPLICADAS EN REGION2
0




In [5]:
for i in range(len(df)):
    df[i].drop(columns=['id'], inplace=True)

In [6]:
printDfInfo()
    

REGION0
         f0        f1        f2     product
0  0.705745 -0.497823  1.221170  105.280062
1  1.334711 -0.340164  4.365080   73.037750
2  1.022732  0.151990  1.419926   85.265647
3 -0.032172  0.139033  2.978566  168.620776
4  1.988431  0.155413  4.751769  154.036647

info
<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 4 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   f0       100000 non-null  float64
 1   f1       100000 non-null  float64
 2   f2       100000 non-null  float64
 3   product  100000 non-null  float64
dtypes: float64(4)
memory usage: 3.1 MB
None


REGION1
          f0         f1        f2     product
0 -15.001348  -8.276000 -0.005876    3.179103
1  14.272088  -3.475083  0.999183   26.953261
2   6.263187  -5.948386  5.001160  134.766305
3 -13.081196 -11.506057  4.999415  137.945408
4  12.702195  -8.147433  5.004363  134.766305

info
<class 'pandas.DataFrame'>
RangeIndex: 100000 entries

## Model Training and Evaluation by Region

### The dataset is partitioned into training and validation subsets using a 75:25 ratio to ensure robust model development and unbiased performance assessment.

In [7]:
def dividirDf(df: pd.DataFrame):
    # We separate the DataFrame into feature variables and target variable
    features = df.drop(['product'], axis=1)
    target = df['product']
    
    # We perform the dataset partitioning
    features_train, features_temp, target_train, target_temp = train_test_split(
        features, target, test_size=0.25, random_state=12345
    )
    
    return features_train, features_temp, target_train, target_temp

### Model Training and Validation Set Prediction Generation

In [8]:
# Model training procedure
def predecirValores(features_train,target_train,features_valid):
    model = LinearRegression()
    model.fit(features_train, target_train)
    predicted_valid = model.predict(features_valid)
    return predicted_valid

### The predicted average reserve volume is computed and presented, followed by the model’s Root Mean Squared Error (RMSE), in order to comprehensively assess predictive performance and overall accuracy.

In [9]:
def mediaRmse(target_valid, predicted_valid):
    predicted_mean = predicted_valid.mean()
    rmse = (mean_squared_error(target_valid, predicted_valid)) ** (0.5)
    return predicted_mean, rmse

### As an additional benchmark, we compare the model’s performance against a baseline approach that relies solely on the mean value, allowing us to determine whether the predictive model outperforms a simple average-based strategy.

In [10]:
def compValid(target_train, target_valid, predicted_valid):
    predicted_valid = pd.Series(target_train.mean(), index = target_valid.index)
    rmse = (mean_squared_error(target_valid, predicted_valid )) ** (0.5)
    return rmse

### The same methodological procedure is systematically applied to each of the three DataFrames to ensure consistency, comparability, and methodological rigor across all regions.

I will construct a list containing the data from each DataFrame organized within dictionaries. This structured approach provides greater clarity, flexibility, and control over the workflow.

In [11]:
resultados = []
for i in range(len(df)):
    resultados.append({})
    resultados[i]["name"] = "Region {}".format(i)
    # Split and partition
    resultados[i]["features_train"], resultados[i]["features_valid"], resultados[i]["target_train"], resultados[i]["target_valid"] = dividirDf(df[i])
    # Generate predictions
    resultados[i]["predicted_valid"] = predecirValores(
        resultados[i]["features_train"],
        resultados[i]["target_train"],
        resultados[i]["features_valid"]
    )
    # Report the predicted mean volume and RMSE
    resultados[i]["predicted_mean"], resultados[i]["rmse"] = mediaRmse(
        resultados[i]["target_valid"],
        resultados[i]["predicted_valid"]
    )
    # Baseline RMSE using the mean
    resultados[i]["rmse_mean"] = compValid(
        resultados[i]["target_train"],
        resultados[i]["target_valid"],
        resultados[i]["predicted_valid"]
    )

In [12]:
df_resultados = pd.DataFrame(resultados)
df_resultados

,name,features_train,features_valid,target_train,target_valid,predicted_valid,predicted_mean,rmse,rmse_mean
0,Region 0,f0 f1 f2 27212 0....,f0 f1 f2 71751 0.9...,27212 147.370612 7866 147.630053 62041 ...,71751 10.038645 80493 114.551489 2655 ...,"[95.89495184592089, 77.57258260954849, 77.8926...",92.592568,37.579422,44.289591
1,Region 1,f0 f1 f2 27212 -...,f0 f1 f2 71751 -...,27212 84.038886 7866 80.859783 62041 ...,71751 80.859783 80493 53.906522 2655 ...,"[82.66331364514762, 54.43178615614918, 29.7487...",68.728547,0.893099,46.021445
2,Region 2,f0 f1 f2 27212 -0.9...,f0 f1 f2 71751 -1.4...,27212 16.733577 7866 38.047492 62041 ...,71751 61.212375 80493 41.850118 2655 ...,"[93.59963303173879, 75.10515853556227, 90.0668...",94.965046,40.029709,44.902350


### Storing the Predictions and Ground Truth Values for the Validation Set

In [13]:
df_pred = df_resultados['predicted_valid'].apply(pd.Series).T.reset_index(drop=True)
df_real = df_resultados['target_valid'].apply(pd.Series).T.reset_index(drop=True)

### At this stage, we carefully examine the results obtained from each region, comparing performance metrics, interpreting the predictive accuracy, and identifying any relevant patterns or differences that may inform our final decision.

In [14]:
df_resultados = df_resultados.drop(df_resultados.columns[1:6], axis=1).reset_index(drop=True)
df_resultados

,name,predicted_mean,rmse,rmse_mean
0,Region 0,92.592568,37.579422,44.289591
1,Region 1,68.728547,0.893099,46.021445
2,Region 2,94.965046,40.029709,44.902350


Our model demonstrates superior performance compared to a naïve baseline approach that relies solely on the mean value to predict the product output.
In the case of Region 1, the RMSE value is notably low. This could either indicate an exceptionally strong predictive fit for that specific region or warrant further verification to rule out potential data inconsistencies or anomalies.
The Root Mean Squared Error values of 37, 1, and 40 represent the average magnitude of prediction error for each respective region. In practical terms, these values quantify how much, on average, our model’s predictions deviate from the actual observed production levels.
For Regions 0 and 2, the error appears relatively high, particularly considering that production is measured in thousands of barrels. From a business perspective, this level of deviation could translate into significant forecasting risk and should be carefully evaluated before making investment decisions.

## Profit Calculation

### We store all the necessary values for the calculations in separate variables to ensure clarity, structure, and transparency in the subsequent profit analysis.

In [15]:
npuntos = 500
nselec = 200
presupuesto = 100000000
pre_barril = 4.5
ingreso_barril = pre_barril * 1000
unmbral = 0.025
state = RandomState(12345)
nsam = 1000

### Next step

Given an investment of 100 million dollars allocated across 200 oil wells, each well must generate at least 500,000 dollars in revenue on average to avoid losses (which is equivalent to 111.1 units of production).
At this stage, we compare this profitability threshold with the average reserve volume in each region to determine whether the projected production levels are sufficient to ensure financial viability.

In [16]:
def compararInv(reserves, i):
    average_reserves = reserves.mean()
    
    if average_reserves > 111.1:
        print(
            "For Region {} the average reserve volume is {}, which exceeds the required threshold.".format(
                i, average_reserves
            )
        )
    else:
        print(
            "For Region {} the average reserve volume is {}, which falls below the required threshold.".format(
                i, average_reserves
            )
        )
    
    print("\n")

In [17]:
for i in range(len(df)):
    compararInv(df[i]["product"], i)

For Region 0 the average reserve volume is 92.5, which falls below the required threshold.


For Region 1 the average reserve volume is 68.825, which falls below the required threshold.


For Region 2 the average reserve volume is 95.0, which falls below the required threshold.




### Presenting Conclusions on the Preparation Phase for Profit Calculation


To calculate profit, we apply the following formula:
Operating Profit = Gross Profit − Operating Expenses.
We first compute the gross profit by multiplying the number of produced barrel units by the revenue per unit, and subsequently subtract the allocated budget to obtain the net result.
Next, we implement a subsampling approach by randomly selecting 500 wells, and from this subset, identifying the top 200 wells with the highest predicted production levels to simulate an optimized investment strategy.

## Writing a function to calculate the profit for a selected set of oil wells, and then modeling the corresponding predictions:

### Selecting the 200 wells with the highest predicted values in each of the three regions

In [18]:
top200 = pd.DataFrame()
for i in df_pred.columns:
    top200[i] = df_pred[i].nlargest(nselec).reset_index(drop=True)
top200

,0,1,2
0,180.180713,139.818970,165.856833
1,176.252213,139.773423,165.679685
2,175.850623,139.703330,163.439962
3,175.658429,139.560938,162.062589
4,173.299686,139.516754,161.797476
...,...,...,...
195,148.507064,138.421423,142.490763
196,148.481767,138.416960,142.485922
197,148.476498,138.413881,142.465777
198,148.436761,138.412834,142.454763


### Summarizing the target reserve volume based on the selected predictions and store the predicted values corresponding to the top 200 wells for each of the three regions.

In [19]:
meanTopPredic = top200.mean()
meanTopPredic

0    155.511654
1    138.730134
2    148.019493
dtype: float64

The established minimum target is 112 units. By selecting the top 200 predicted wells, we anticipate achieving production levels that exceed this threshold and, consequently, generating a positive profit margin.

### We proceed by calculating the potential profit generated by the top 200 wells in each region, using the previously defined revenue and cost structure. This allows us to quantify the projected financial return under an optimized selection strategy.

After evaluating the results, we compare the total projected profits across all three regions, taking into account both the magnitude of returns and the associated predictive reliability.
Based on this comparative analysis, the region with the highest positive projected profit and acceptable risk profile should be recommended for oil field development. The final selection is justified by its superior expected return, its ability to exceed the minimum production threshold, and the relative stability observed in the model’s predictive performance.

We do not need to multiply every individual value in each column by the profit per barrel. Instead, we can multiply the average value by the number of rows to estimate the total number of barrels, and then multiply that total by the revenue per unit.

In [20]:
ventas = meanTopPredic * nselec * pre_barril
gastos = presupuesto / 1000
gananciaTop = ventas - gastos
gananciaTop

0    39960.488775
1    24857.120520
2    33217.543962
dtype: float64

We also account for the fact that we must multiply by 1,000 to obtain the true profit value.
At this stage, we cannot make a definitive decision because the exercise does not specify the priority criterion whether the focus is risk reduction, maximum profit, expected return on investment, or another objective.

If the goal were to maximize profit, we would select Region 0, since it presents the highest estimated oil volume and therefore generates the greatest projected profit. In addition, its RMSE is comparable to that of Region 2.

## Calculating both the projected profits and the associated risks for each region, enabling a balanced evaluation that considers not only potential returns but also the variability and uncertainty

### Using the predictions stored in last steps, we apply the bootstrapping technique with 1,000 resamples in order to estimate the distribution of potential profits and assess the variability and risk associated with each region.

In [21]:
df_pred

,0,1,2
0,95.894952,82.663314,93.599633
1,77.572583,54.431786,75.105159
2,77.892640,29.748760,90.066809
3,90.175134,53.552133,105.162375
4,70.510088,1.243856,115.303310
...,...,...,...
24995,103.037104,136.869211,78.765887
24996,85.403255,110.693465,95.603394
24997,61.509833,137.879341,99.407281
24998,118.180397,83.761966,77.779912


In [22]:
df_real

,0,1,2
0,10.038645,80.859783,61.212375
1,114.551489,53.906522,41.850118
2,132.603635,30.132364,57.776581
3,169.072125,53.906522,100.053761
4,122.325180,0.000000,109.897122
...,...,...,...
24995,170.116726,137.945408,28.492402
24996,93.632175,110.992147,21.431303
24997,127.352259,137.945408,125.487229
24998,99.782700,84.038886,99.422903


In [23]:
def muestraTop200(serie, npuntos, nselec):
    subsample = serie.sample(n= npuntos, replace=True, random_state=state)
    topValues= subsample.nlargest(nselec)
    return topValues
def productoReal(top, reales):
    indices = top.index
    real_values = reales.loc[indices]
    return real_values

### Computing the average profit, construct the 95% confidence interval, and estimate the risk of losses.
A loss is defined as a negative profit outcome; therefore, we calculate the probability of obtaining a negative value within the simulated distribution and then express this probability as a percentage to quantify the overall risk exposure.

In [24]:
def beneficio(serie):
    beneficio = (serie.sum() * pre_barril) - (presupuesto / 1000)
    return beneficio
def intervConfianza(values):
    lower = values.quantile(0.025)
    upper = values.quantile(0.975)
    return lower, upper

In [25]:
values= pd.DataFrame(columns=df_pred.columns)
final = pd.DataFrame(index=range(3), columns=df_pred.columns)
for col in df_pred:
    for i in range(nsam):
        topValues = muestraTop200(df_pred[col], npuntos, nselec)
        real_values = productoReal(topValues, df_real[col])
        benef = beneficio(real_values)
        values.loc[i, col] = benef
    lower, upper = intervConfianza(values[col])
    benefPromedio = values[col].mean()
    final[col] = [lower, benefPromedio, upper]
final.columns = ['Region 0', 'Region 1', 'Region 2']
final.index = ['Lower', 'media', 'Upper']
final

,Region 0,Region 1,Region 2
Lower,-1112.155459,780.508108,-1122.276254
media,3961.649848,4611.558173,3929.504752
Upper,9097.669416,8629.520603,9345.629146


#### Loss risk

In [26]:
(values < 0).sum() / values.size * 100


0    2.300000
1    0.233333
2    2.166667
dtype: float64

### Presenting our conclusions by recommending a region for oil well development and justifying our choice based on the average profit, the 95% confidence interval, and the risk of losses.

Finally, we verify whether this recommendation aligns with the earlier selection made in Section 4.3, and we briefly explain any differences if the chosen region changes under the risk-based evaluation.

Evidently, Region 1 is the best option: it delivers the highest expected value (mean profit), a narrower 95% confidence interval, and the lowest risk of loss.
The final decision depends on the level of risk we are willing to assume. Although Region 1 initially appeared less attractive, the bootstrapping analysis provides a more reliable view of uncertainty and indicates that it is the strongest choice under a risk-aware evaluation framework.
Therefore, Region 1 is the recommended region for development.